<a href="https://colab.research.google.com/github/Jaeji/AM360Paper/blob/main/Model_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. ติดตั้งและ Import Library

In [1]:
!pip install -q kagglehub xgboost scikit-learn pandas numpy scipy

import numpy as np
import pandas as pd
import os, time, warnings
import kagglehub
from scipy.io import loadmat
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Lasso
from sklearn.kernel_ridge import KernelRidge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# 2. เลือกแหล่งข้อมูลและโหลด Data

In [2]:
# --- ตัวเลือกแหล่งข้อมูล ---
DATA_SOURCE = int(input("เลือกแหล่งข้อมูล (1: NASA, 2: Kaggle): "))

while DATA_SOURCE not in [1, 2]:
  DATA_SOURCE = int(input("เลือกแหล่งข้อมูล (1: NASA, 2: Kaggle): "))

if DATA_SOURCE == 1:
  DATA_SOURCE = 'NASA'
if DATA_SOURCE ==2:
  DATA_SOURCE = 'KAGGLE'

# -------------------------

def load_nasa_data():
    if not os.path.exists('AM360Paper'):
        !git clone https://github.com/Jaeji/AM360Paper.git

    def extract(mat_path, b_name):
        mat = loadmat(mat_path)
        d = mat[b_name][0, 0]['cycle'][0]
        data = []
        for i in range(len(d)):
            if d[i]['type'][0] == 'discharge':
                dd = d[i]['data'][0, 0]
                if dd['Capacity'][0,0] > 0:
                    data.append([
                        np.mean(dd['Voltage_measured']), np.std(dd['Voltage_measured']),
                        np.mean(dd['Current_measured']), np.std(dd['Current_measured']),
                        np.mean(dd['Temperature_measured']), np.std(dd['Temperature_measured']),
                        dd['Capacity'][0,0]
                    ])
        return pd.DataFrame(data, columns=['V_mean', 'V_std', 'I_mean', 'I_std', 'Temp_mean', 'Temp_std', 'Capacity'])

    # โหลด 4 ก้อน
    batteries = ["B0005", "B0006", "B0007", "B0018"]
    all_df = pd.concat([extract(f"AM360Paper/{b}.mat", b) for b in batteries if os.path.exists(f"AM360Paper/{b}.mat")])
    # สร้าง RUL จาก Capacity (หรือใช้ Index ย้อนกลับในขั้นตอนถัดไป)
    all_df['RUL'] = np.arange(len(all_df))[::-1]
    return all_df, ['V_mean', 'V_std', 'I_mean', 'I_std', 'Temp_mean', 'Temp_std', 'Capacity'], 'RUL'

def load_kaggle_data():
    # โหลดจาก Kaggle
    path = kagglehub.dataset_download("ignaciovinuales/battery-remaining-useful-life-rul")
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    df = pd.read_csv(os.path.join(path, csv_file))

    # เลือก Features อัตโนมัติ (ตัดคอลัมน์ที่ไม่ใช่ออก)
    exclude_cols = ['RUL', 'Cycle_Index', 'Battery_ID']
    feats = [c for c in df.columns if c not in exclude_cols]
    return df, feats, 'RUL'

# เลือกโหลดตามตัวแปร DATA_SOURCE
if DATA_SOURCE == "NASA":
    df, features, target = load_nasa_data()
    print(f"Loaded NASA Data: {df.shape}")
else:
    df, features, target = load_kaggle_data()
    print(f"Loaded Kaggle Data: {df.shape}")

เลือกแหล่งข้อมูล (1: NASA, 2: Kaggle): 2


100%|██████████| 374k/374k [00:00<00:00, 62.0MB/s]

Extracting files...
Loaded Kaggle Data: (15064, 9)


# 3. เตรียมข้อมูลและ Train/Test Split

In [3]:
X = df[features].values
y = df[target].values

# Scale ข้อมูล
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 4. กำหนดโมเดลและเริ่มการทดสอบ

In [4]:
N_EST = 200  # กำหนดตัวแปรกลางสำหรับจำนวนรอบ (Estimators)

models = {
    "Lasso": Lasso(alpha=0.01),
    "Kernel Ridge": KernelRidge(alpha=0.1, kernel="rbf"),
    "Decision Tree": DecisionTreeRegressor(max_depth=8, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=N_EST, max_depth=12, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=N_EST, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=N_EST, learning_rate=0.1, max_depth=8, random_state=42)
}

# รัน Loop เก็บค่าทั้งหมด
results_data = []
for name, model in models.items():
    start_t = time.time()
    model.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)
    time_taken = time.time() - start_t
    y_train_pred = model.predict(X_train)

    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    sse_train = np.sum((y_train - y_train_pred)**2)
    sse_test = np.sum((y_test - y_test_pred)**2)
    tot_train = np.sum(np.abs(y_train - y_train_pred))
    tot_test = np.sum(np.abs(y_test - y_test_pred))
    rel_train = np.mean(np.abs(y_train - y_train_pred) / (y_train + 1))
    rel_test = np.mean(np.abs(y_test - y_test_pred) / (y_test + 1))

    results_data.append([name, rmse_train, rmse_test, sse_train, sse_test, tot_train, tot_test, rel_train, rel_test, time_taken])

df_res = pd.DataFrame(results_data, columns=[
    "Methods", "RMSE Train", "RMSE Test", "SSE Train", "SSE Test",
    "Total Error Train", "Total Error Test", "Relative Error Train", "Relative Error Test", "Time (second)"
])

# 5. สร้างตารางสรุปตาม Paper

In [5]:
def show_section_5_table(df, cols, title, table_num):
    subset = df[cols].copy()

    styles = [
        {'selector': 'th', 'props': [('border-top', '3px double black'), ('border-bottom', '1px solid black'),
                                     ('font-family', 'Times New Roman, serif'), ('font-size', '11pt'),
                                     ('text-align', 'center'), ('padding', '8px'), ('font-weight', 'bold'),
                                     ('background-color', 'white'), ('color', 'black')]},
        {'selector': 'td', 'props': [('font-family', 'Times New Roman, serif'), ('font-size', '11pt'),
                                     ('text-align', 'center'), ('padding', '6px'), ('background-color', 'white'), ('color', 'black')]},
        {'selector': 'tr:last-child td', 'props': [('border-bottom', '2px solid black')]},
        {'selector': 'caption', 'props': [('caption-side', 'top'), ('text-align', 'center'), ('font-weight', 'bold'),
                                          ('font-size', '12pt'), ('font-family', 'Times New Roman, serif'), ('margin-bottom', '10px'), ('color', 'black')]}
    ]
    caption = f"TABLE {table_num}<br>{title}"
    display(HTML(subset.style.set_table_styles(styles).set_caption(caption).format(precision=4).hide(axis='index').to_html()))
    print("<br>")

print("EXPERIMENTAL RESULTS\n")

# Table II
df_sorted = df_res.sort_values("RMSE Test")
show_section_5_table(df_sorted, ["Methods", "RMSE Train", "RMSE Test", "SSE Train", "SSE Test"],
                     "RMSE AND SSE VALUES OF RUL ESTIMATION MODELS", "II")

# Table III
df_sorted = df_res.sort_values("Total Error Test")
show_section_5_table(df_sorted, ["Methods", "Total Error Train", "Total Error Test"],
                     "TOTAL ERROR VALUES OF RUL ESTIMATION MODELS", "III")

# Table IV
df_sorted = df_res.sort_values("Relative Error Test")
show_section_5_table(df_sorted, ["Methods", "Relative Error Train", "Relative Error Test"],
                     "AVERAGE RELATIVE ERROR VALUES OF RUL ESTIMATION MODELS", "IV")

# Table V
df_sorted = df_res.sort_values("Time (second)")
show_section_5_table(df_sorted, ["Methods", "Time (second)"],
                     "RUNNING TIMES OF RUL MODELS", "V")

EXPERIMENTAL RESULTS



Methods,RMSE Train,RMSE Test,SSE Train,SSE Test
Random Forest,12.7938,29.2957,1972512.7315,2585870.3148
XGBoost,8.4143,30.9037,853216.8911,2877533.9212
Gradient Boosting,35.7433,41.4167,15396198.6575,5168324.5643
Decision Tree,31.7758,43.5663,12167926.9194,5718742.5095
Kernel Ridge,50.0976,64.2116,30245252.4265,12422971.0756
Lasso,142.0135,163.5788,243042623.7105,80621934.2994


<br>


Methods,Total Error Train,Total Error Test
XGBoost,64944.7259,37419.8168
Random Forest,91974.8798,41662.0647
Decision Tree,263622.5833,78593.9694
Gradient Boosting,317107.5272,86666.7330
Kernel Ridge,453432.6128,118791.6843
Lasso,1096552.7000,289828.1363


<br>


Methods,Relative Error Train,Relative Error Test
Kernel Ridge,0.3002,0.3692
Gradient Boosting,0.2067,0.4538
Random Forest,0.0883,0.5388
XGBoost,0.0336,0.5576
Decision Tree,0.2010,0.6054
Lasso,0.8918,1.3262


<br>


Methods,Time (second)
Lasso,0.0179
Decision Tree,0.1007
XGBoost,4.3812
Gradient Boosting,7.3516
Random Forest,16.0126
Kernel Ridge,34.9513


<br>
